In [1]:
import pandas as pd
from rapidfuzz import process, fuzz

In [2]:
categories = pd.read_parquet(r'../export/small_parquet/municiplity.parquet')
categories

,DE_MUNICIP
0,ABRERA
1,ACEBEDO
2,ADEJE CASCO
3,AGUILON
4,ALAMEDA DE LA SAGRA
...,...
804,VILLASILA DE VALDAVIA
805,VILLAZANZO DE VALDERADUEY
806,VITORIA-GASTEIZ
807,VIVEIRO


In [3]:
categories.nunique()

DE_MUNICIP    809
dtype: int64

In [4]:
categories.dtypes

DE_MUNICIP    object
dtype: object

In [5]:
categories['DE_MUNICIP'] = categories['DE_MUNICIP'].str.strip()

In [6]:
categories.sort_values(by='DE_MUNICIP', inplace=True)
categories.head()

,DE_MUNICIP
201,
202,A CORUÃ‘A
203,ABIA DE LAS TORRES
0,ABRERA
1,ACEBEDO


In [7]:
categories.isna().sum()

DE_MUNICIP    0
dtype: int64

In [8]:
categories = categories[categories['DE_MUNICIP'].str.strip() != '']
categories.shape

(808, 1)

In [12]:
latlong_data = pd.read_csv(r"..\BD_MUNICIPIOS-ENTIDADES\MUNICIPIOS.csv",
                           delimiter=';',
                           encoding='latin1')
latlong_data.head()

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD
0,1001000000,1010014,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,0113-3,"-2,512507724","42,84045247",Detección automática,568,MDT
1,1002000000,1010029,1020,1,Araba/Álava,Amurrio,10346,"9617,86",65701,1002000201,Amurrio,9256,0086-4,"-3,001015194","43,05265767",Detección automática,217,MDT
2,1003000000,1010035,1030,1,Araba/Álava,Aramaio,1353,"7308,96",42097,1003000601,Ibarra,731,0087-4,"-2,564829379","43,05257873",Detección automática,325,MDT
3,1004000000,1010040,1040,1,Araba/Álava,Artziniega,1868,"2728,73",22886,1004000101,Artziniega,1732,0086-1,"-3,13052099","43,1217919",Detección automática,199,MDT
4,1006000000,1010066,1060,1,Araba/Álava,Armiñón,233,"1297,27",24707,1006000101,Armiñón,106,0137-4,"-2,872270813","42,72340924",Detección automática,466,MDT


In [13]:
latlong_data = latlong_data[['NOMBRE_ACTUAL', 'PROVINCIA', 'LONGITUD_ETRS89_REGCAN95', 'LATITUD_ETRS89_REGCAN95']]
latlong_data.shape

(8132, 4)

In [14]:
latlong_data.rename(columns={
    'NOMBRE_ACTUAL':'DE_MUNICIP',
    'PROVINCIA':'PROVINCE',
    'LONGITUD_ETRS89_REGCAN95':'LONGITUDE',
    'LATITUD_ETRS89_REGCAN95':'LATITUDE'
},
inplace=True)
latlong_data.head()

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,Alegría-Dulantzi,Araba/Álava,"-2,512507724","42,84045247"
1,Amurrio,Araba/Álava,"-3,001015194","43,05265767"
2,Aramaio,Araba/Álava,"-2,564829379","43,05257873"
3,Artziniega,Araba/Álava,"-3,13052099","43,1217919"
4,Armiñón,Araba/Álava,"-2,872270813","42,72340924"


In [15]:
latlong_data.dtypes

DE_MUNICIP    object
PROVINCE      object
LONGITUDE     object
LATITUDE      object
dtype: object

In [16]:
latlong_data['DE_MUNICIP'] = latlong_data['DE_MUNICIP'].str.strip().str.upper()
latlong_data['PROVINCE'] = latlong_data['PROVINCE'].str.strip()
latlong_data['LATITUDE'] = latlong_data['LATITUDE'].str.replace(',', '.').astype('float64')
latlong_data['LONGITUDE'] = latlong_data['LONGITUDE'].str.replace(',', '.').astype('float64')
print(latlong_data.dtypes)
latlong_data.head()

DE_MUNICIP     object
PROVINCE       object
LONGITUDE     float64
LATITUDE      float64
dtype: object


,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,ALEGRÍA-DULANTZI,Araba/Álava,-2.512508,42.840452
1,AMURRIO,Araba/Álava,-3.001015,43.052658
2,ARAMAIO,Araba/Álava,-2.564829,43.052579
3,ARTZINIEGA,Araba/Álava,-3.130521,43.121792
4,ARMIÑÓN,Araba/Álava,-2.872271,42.723409


In [17]:
latlong_data.sort_values(by='DE_MUNICIP', inplace=True)
latlong_data.head()

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
4891,A ARNOIA,Ourense,-8.135751,42.253066
2132,A BAÑA,A Coruña,-8.758003,42.961898
4902,A BOLA,Ourense,-7.928518,42.140937
2143,A CAPELA,A Coruña,-8.068806,43.435552
5292,A CAÑIZA,Pontevedra,-8.273359,42.212754


In [18]:
merged = categories.merge(latlong_data, on=['DE_MUNICIP'], how='left')
merged

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,A CORUÃ‘A,NaN,NaN,NaN
1,ABIA DE LAS TORRES,Palencia,-4.421902,42.420220
2,ABRERA,Barcelona,1.901569,41.516398
3,ACEBEDO,León,-5.115871,43.040732
4,ACEUCHAL,Badajoz,-6.487251,38.647755
...,...,...,...,...
809,YUNQUERA,Málaga,-4.917332,36.734203
810,ZAFRA,Badajoz,-6.418559,38.426344
811,ZARAGOZA,Zaragoza,-0.877318,41.656208
812,ZARZOSA DE RIO PISUERGA,NaN,NaN,NaN


In [19]:
merged[merged['LONGITUDE'].isna()]

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,A CORUÃ‘A,NaN,NaN,NaN
6,ADEJE CASCO,NaN,NaN,NaN
10,AGUILON,NaN,NaN,NaN
11,AGUIMES,NaN,NaN,NaN
12,AGÃœIMES,NaN,NaN,NaN
...,...,...,...,...
798,VILLAYUSO,NaN,NaN,NaN
800,VILOBI D'ONYAR,NaN,NaN,NaN
801,VINAROS,NaN,NaN,NaN
805,XUNQUEIRA DE AMBIA,NaN,NaN,NaN


In [20]:
merged[merged['DE_MUNICIP'].duplicated(keep=False)]

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
131,CABANES,Castelló/Castellón,0.045412,40.156104
132,CABANES,Girona,2.977957,42.307504
199,CIEZA,Murcia,-1.427730,38.236594
200,CIEZA,Cantabria,-4.096722,43.221056
433,MIERES,Girona,2.640292,42.123296
434,MIERES,Asturias,-5.772657,43.248794
579,SADA,Navarra,-1.397668,42.585825
580,SADA,A Coruña,-8.253590,43.351531
718,TORRENT,València/Valencia,-0.465982,39.436721
719,TORRENT,Girona,3.128197,41.951843


In [29]:
categories_prov = pd.read_parquet(r'..\export\small_parquet\muncipility_province.parquet')
print(categories_prov.shape)
categories_prov.head()

(778, 2)


,Municipality,Province
0,Abla,Almería
1,Abrucena,Almería
2,Adamuz,Córdoba
3,Adra,Almería
4,Agrón,Granada


In [22]:
same_name_diff_prov = merged[merged['DE_MUNICIP'].duplicated()]['DE_MUNICIP'].values
same_name_diff_prov

array(['CABANES', 'CIEZA', 'MIERES', 'SADA', 'TORRENT', 'VILLAESCUSA'],
      dtype=object)

In [23]:
categories_prov[categories_prov['Municipality'].isin(same_name_diff_prov)]

,Municipality,Province


In [24]:
dataset_districts = categories.DE_MUNICIP.tolist()
all_districts = latlong_data['DE_MUNICIP'].tolist()

matches = {}
for name in dataset_districts:
    result = process.extractOne(
        name,
        all_districts,
        scorer=fuzz.token_sort_ratio,  
        score_cutoff=50,               
    )
    if result:
        matches[name] = (result[0], result[1]) 
    else:
        matches[name] = (None, 0)

review_all = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in matches.items()],
    columns=["DE_MUNICIP_org", "DE_MUNICIP", "score"],
).sort_values("score", ascending=False)
print(review_all.to_string())

                     DE_MUNICIP_org                                  DE_MUNICIP       score
807                           ZUERA                                       ZUERA  100.000000
805                        ZARAGOZA                                    ZARAGOZA  100.000000
1                ABIA DE LAS TORRES                          ABIA DE LAS TORRES  100.000000
2                            ABRERA                                      ABRERA  100.000000
3                           ACEBEDO                                     ACEBEDO  100.000000
4                          ACEUCHAL                                    ACEUCHAL  100.000000
789                  VILLASARRACINO                              VILLASARRACINO  100.000000
788                    VILLASANDINO                                VILLASANDINO  100.000000
787            VILLARTA DE SAN JUAN                        VILLARTA DE SAN JUAN  100.000000
786         VILLARRUBIA DE LOS OJOS                     VILLARRUBIA DE LOS OJOS 

In [25]:
len(review_all)

808

In [26]:
merged_all = review_all.merge(latlong_data, on='DE_MUNICIP', how='left')
merged_all.head()

,DE_MUNICIP_org,DE_MUNICIP,score,PROVINCE,LONGITUDE,LATITUDE
0,ZUERA,ZUERA,100.0,Zaragoza,-0.787165,41.868393
1,ZARAGOZA,ZARAGOZA,100.0,Zaragoza,-0.877318,41.656208
2,ABIA DE LAS TORRES,ABIA DE LAS TORRES,100.0,Palencia,-4.421902,42.420220
3,ABRERA,ABRERA,100.0,Barcelona,1.901569,41.516398
4,ACEBEDO,ACEBEDO,100.0,León,-5.115871,43.040732


In [27]:
merged_all.isna().sum()

DE_MUNICIP_org    0
DE_MUNICIP        0
score             0
PROVINCE          0
LONGITUDE         0
LATITUDE          0
dtype: int64

In [28]:
merged_all[merged_all['score']<70]

,DE_MUNICIP_org,DE_MUNICIP,score,PROVINCE,LONGITUDE,LATITUDE
797,SANT CARLES DE LA RAPITA,SANTA CRUZ DE LA PALMA,69.565217,Santa Cruz de Tenerife,-17.764703,28.683343
798,PUENTENANSA,FUENTECANTOS,69.565217,Soria,-2.428761,41.849387
799,VILLARREAL/VILA-REAL,VILLARRAMIEL,68.750000,Palencia,-4.911435,42.042614
800,LAS PALMAS DE G.C.,LAS PALMAS DE GRAN CANARIA,68.181818,Las Palmas,-15.428470,28.124924
801,RUBAYO,BAREYO,66.666667,Cantabria,-3.612028,43.479111
802,AJO,ARJONA,66.666667,Jaén,-4.055308,37.936555
803,MIÃ‘O,MIÑO,66.666667,A Coruña,-8.204915,43.347300
804,SAN ILDEFONSO,SAN ASENSIO,66.666667,La Rioja,-2.748913,42.497434
805,ELEXALDE,L'ALDEA,66.666667,Tarragona,0.620134,40.745766
806,ELCHE/ELX,LEACHE/LEATXE,63.636364,Navarra,-1.408207,42.606840


In [32]:
merged_all.to_csv(r'../BD_MUNICIPIOS-ENTIDADES/DE_MUNICIP_LAT_LONG.csv')